In [1]:
## Our first task is to understand the dataset we have been given
## A deep understanding of the nature, structure, and shortcoming of this data
## will enable a more streamlined and powerful ML pipeline later on

import pandas as pd

file_path = "/Users/stephenwillis/Desktop/archive/SBAnational.csv" ## CHANGE TO YOUR PATH
df = pd.read_csv(file_path)
pd.set_option('display.max_columns', None)
df.head()

/var/folders/fk/1xxdjzqs77g4r4zq8qj557rw0000gn/T/ipykernel_1569/2849486171.py:8: DtypeWarning: Columns (9) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path)


,LoanNr_ChkDgt,Name,City,State,Zip,Bank,BankState,NAICS,ApprovalDate,ApprovalFY,Term,NoEmp,NewExist,CreateJob,RetainedJob,FranchiseCode,UrbanRural,RevLineCr,LowDoc,ChgOffDate,DisbursementDate,DisbursementGross,BalanceGross,MIS_Status,ChgOffPrinGr,GrAppv,SBA_Appv
0,1000014003,ABC HOBBYCRAFT,EVANSVILLE,IN,47711,FIFTH THIRD BANK,OH,451120,28-Feb-97,1997,84,4,2.0,0,0,1,0,N,Y,NaN,28-Feb-99,"$60,000.00",$0.00,P I F,$0.00,"$60,000.00","$48,000.00"
1,1000024006,LANDMARK BAR & GRILLE (THE),NEW PARIS,IN,46526,1ST SOURCE BANK,IN,722410,28-Feb-97,1997,60,2,2.0,0,0,1,0,N,Y,NaN,31-May-97,"$40,000.00",$0.00,P I F,$0.00,"$40,000.00","$32,000.00"
2,1000034009,"WHITLOCK DDS, TODD M.",BLOOMINGTON,IN,47401,GRANT COUNTY STATE BANK,IN,621210,28-Feb-97,1997,180,7,1.0,0,0,1,0,N,N,NaN,31-Dec-97,"$287,000.00",$0.00,P I F,$0.00,"$287,000.00","$215,250.00"
3,1000044001,"BIG BUCKS PAWN & JEWELRY, LLC",BROKEN ARROW,OK,74012,1ST NATL BK & TR CO OF BROKEN,OK,0,28-Feb-97,1997,60,2,1.0,0,0,1,0,N,Y,NaN,30-Jun-97,"$35,000.00",$0.00,P I F,$0.00,"$35,000.00","$28,000.00"
4,1000054004,"ANASTASIA CONFECTIONS, INC.",ORLANDO,FL,32801,FLORIDA BUS. DEVEL CORP,FL,0,28-Feb-97,1997,240,14,1.0,7,7,1,0,N,N,NaN,14-May-97,"$229,000.00",$0.00,P I F,$0.00,"$229,000.00","$229,000.00"


In [2]:
## Right away we see there is an problem, namely that in column 9 we have a type issue
## Now we get the name of the column with the blatant type issue
column_nine = df.columns[9]
print(f"Column 9 is: {column_nine}")

## We can also see what types this feature is composed of
print(df[column_nine].apply(type).value_counts())

Column 9 is: ApprovalFY
ApprovalFY
<class 'int'>    768092
<class 'str'>    131072
Name: count, dtype: int64


In [3]:
## From the data documentation, 'ApprovalFY' denotes the year of fiscal commitment
## Therefore, this should be an int, not a string

## We can try simply typecasting and see if that works..
typecast_mask = pd.to_numeric(df[column_nine], errors='coerce').isna()
print(f"The # of rows that could not be converted str->int is: {typecast_mask.sum()}")

The # of rows that could not be converted str->int is: 18


In [4]:
## Okay, so it seems that pandas may have simply typecast most of the str values incorrectly
## Likely fixable by simply calling pd.read_csv with low_memory=False
## Nonetheless, there are still 18 problem rows, we could get away with simply dropping
## But at this point we might as well finish our investigation

print(df.loc[typecast_mask, column_nine])

699732    1976A
704030    1976A
705375    1976A
710381    1976A
713245    1976A
748029    1976A
751519    1976A
769515    1976A
775002    1976A
775430    1976A
775978    1976A
776367    1976A
780120    1976A
781090    1976A
784351    1976A
788539    1976A
788661    1976A
793733    1976A
Name: ApprovalFY, dtype: object


In [5]:
## It looks like there was simply a data entry issue for some entries in this year
## We can feel quite confident about this assumption if these values match up
## closely with the associated values from ApprovalDate..

print(df.loc[typecast_mask, [column_nine, "ApprovalDate"]])

       ApprovalFY ApprovalDate
699732      1976A    29-Sep-76
704030      1976A    17-Sep-76
705375      1976A    10-Sep-76
710381      1976A    19-Jul-76
713245      1976A    30-Aug-76
748029      1976A    20-Aug-76
751519      1976A    29-Sep-76
769515      1976A    19-Jul-76
775002      1976A    20-Aug-76
775430      1976A    26-Aug-76
775978      1976A     1-Sep-76
776367      1976A    15-Jul-76
780120      1976A    29-Jul-76
781090      1976A    21-Jul-76
784351      1976A    19-Aug-76
788539      1976A    30-Sep-76
788661      1976A    14-Sep-76
793733      1976A    24-Sep-76


In [6]:
## Okay, given the alignment between these two columns, it should be safe to
## simply clean these 18 data points rather than discarding them
## We can clean these points while also converting the other incorrectly
## typecast values in one shot
df[column_nine] = pd.to_numeric(df[column_nine].astype(str).str.extract(r"(\d+)", expand=False), errors = "coerce")

print(f"Post-fix value counts: {df[column_nine].apply(type).value_counts()}")
df.loc[typecast_mask, column_nine]

Post-fix value counts: ApprovalFY
<class 'int'>    899164
Name: count, dtype: int64


699732    1976
704030    1976
705375    1976
710381    1976
713245    1976
748029    1976
751519    1976
769515    1976
775002    1976
775430    1976
775978    1976
776367    1976
780120    1976
781090    1976
784351    1976
788539    1976
788661    1976
793733    1976
Name: ApprovalFY, dtype: int64

In [7]:

## We should make sure to remove all columns 
## that leak information about our target variable ('MIS_Status')
## And speaking of our target variable, it is currently not in a nice form
## Let us address both of these issues before moving forward
## As a reminder, the features and their descriptions can be found in the INTRO notebook.

## By inspection, the columns that clearly leak information are:
## 'ChgOffDate', 'ChgOffPrinGr' 'BalanceGross', 'CreateJob', and RetainedJob'
## We can also drop columns that provide no predictive value
## Although it can be hard to know ahead of time which columns will not be predictive
## At this stage we can safely drop the 'Name' column
## Therefore..
columns_to_drop = ['ChgOffDate', 'BalanceGross', 'ChgOffPrinGr', 'CreateJob', 'RetainedJob', 'Name', 'DisbursementDate']
df = df.drop(axis=1, columns=columns_to_drop)

In [8]:
## Additionally, our target variable could look nicer
## While we are at it, we might as well fix another badly named column
col_rename_dict = {'MIS_Status': 'Defaulted', 'LoanNr_ChkDgt': 'Id'}
df = df.rename(columns = col_rename_dict)
default_map = {'P I F':0, 'CHGOFF': 1}
df['Defaulted'] = df['Defaulted'].map(default_map)
pd.set_option('display.max_columns', None)
df.head()
        


,Id,City,State,Zip,Bank,BankState,NAICS,ApprovalDate,ApprovalFY,Term,NoEmp,NewExist,FranchiseCode,UrbanRural,RevLineCr,LowDoc,DisbursementGross,Defaulted,GrAppv,SBA_Appv
0,1000014003,EVANSVILLE,IN,47711,FIFTH THIRD BANK,OH,451120,28-Feb-97,1997,84,4,2.0,1,0,N,Y,"$60,000.00",0.0,"$60,000.00","$48,000.00"
1,1000024006,NEW PARIS,IN,46526,1ST SOURCE BANK,IN,722410,28-Feb-97,1997,60,2,2.0,1,0,N,Y,"$40,000.00",0.0,"$40,000.00","$32,000.00"
2,1000034009,BLOOMINGTON,IN,47401,GRANT COUNTY STATE BANK,IN,621210,28-Feb-97,1997,180,7,1.0,1,0,N,N,"$287,000.00",0.0,"$287,000.00","$215,250.00"
3,1000044001,BROKEN ARROW,OK,74012,1ST NATL BK & TR CO OF BROKEN,OK,0,28-Feb-97,1997,60,2,1.0,1,0,N,Y,"$35,000.00",0.0,"$35,000.00","$28,000.00"
4,1000054004,ORLANDO,FL,32801,FLORIDA BUS. DEVEL CORP,FL,0,28-Feb-97,1997,240,14,1.0,1,0,N,N,"$229,000.00",0.0,"$229,000.00","$229,000.00"


In [9]:
## Now we should make sure there are no other datatype issues in the dataset
## We can check which columns have more than one type... 

multitype_cols = []
for col in df.columns:
    types = df[col].apply(type).value_counts()
    if types.shape[0] != 1:
        multitype_cols.append(col)
        print(types)
        print("\n")

City
<class 'str'>      899134
<class 'float'>        30
Name: count, dtype: int64


State
<class 'str'>      899150
<class 'float'>        14
Name: count, dtype: int64


Bank
<class 'str'>      897605
<class 'float'>      1559
Name: count, dtype: int64


BankState
<class 'str'>      897598
<class 'float'>      1566
Name: count, dtype: int64


RevLineCr
<class 'str'>      894636
<class 'float'>      4528
Name: count, dtype: int64


LowDoc
<class 'str'>      896582
<class 'float'>      2582
Name: count, dtype: int64




In [10]:
## From inspecting the output above, as well as remembering the expected form of the features,
## we can infer that the 'float' type arises here not from a data entry mistake, but rather
## due to the presence of NaN values (which are floats).  We can confirm this

for col in multitype_cols:
    print(f"All floats are NaN for {col}: {df.loc[(df[col].map(type) == 'float'), col].isna().all()}")

All floats are NaN for City: True
All floats are NaN for State: True
All floats are NaN for Bank: True
All floats are NaN for BankState: True
All floats are NaN for RevLineCr: True
All floats are NaN for LowDoc: True


In [11]:
## Great, there are no further type issues!
## ...
## On the topic of types, we can do some further cleaning and make some of the types more useful
df.describe()

,Id,Zip,NAICS,ApprovalFY,Term,NoEmp,NewExist,FranchiseCode,UrbanRural,Defaulted
count,8.991640e+05,899164.000000,899164.000000,899164.000000,899164.000000,899164.000000,899028.000000,899164.000000,899164.000000,897167.000000
mean,4.772612e+09,53804.391241,398660.950146,2001.143560,110.773078,11.411353,1.280404,2753.725933,0.757748,0.175617
std,2.538175e+09,31184.159152,263318.312759,5.913846,78.857305,74.108196,0.451750,12758.019136,0.646436,0.380494
min,1.000014e+09,0.000000,0.000000,1962.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,2.589758e+09,27587.000000,235210.000000,1997.000000,60.000000,2.000000,1.000000,1.000000,0.000000,0.000000
50%,4.361439e+09,55410.000000,445310.000000,2002.000000,84.000000,4.000000,1.000000,1.000000,1.000000,0.000000
75%,6.904627e+09,83704.000000,561730.000000,2006.000000,120.000000,10.000000,2.000000,1.000000,1.000000,0.000000
max,9.996003e+09,99999.000000,928120.000000,2014.000000,569.000000,9999.000000,2.000000,99999.000000,2.000000,1.000000


In [12]:
## Specifically, ApprovalDate would be better as a datetime object, 
## additionally, DisbursementGross, GrAppv, and SBA_Appv can be cleaned
## via dropping the dollar sign and turned into numeric types
import datetime as dt
df["ApprovalDate"] = pd.to_datetime(df["ApprovalDate"]) #format="%d-%b-%y"

money_cols = ["DisbursementGross", "GrAppv", "SBA_Appv"]
df[money_cols] = df[money_cols].replace(r"[\$,]", "", regex=True).astype(float)

df.head()

/var/folders/fk/1xxdjzqs77g4r4zq8qj557rw0000gn/T/ipykernel_1569/3573865799.py:5: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["ApprovalDate"] = pd.to_datetime(df["ApprovalDate"]) #format="%d-%b-%y"


,Id,City,State,Zip,Bank,BankState,NAICS,ApprovalDate,ApprovalFY,Term,NoEmp,NewExist,FranchiseCode,UrbanRural,RevLineCr,LowDoc,DisbursementGross,Defaulted,GrAppv,SBA_Appv
0,1000014003,EVANSVILLE,IN,47711,FIFTH THIRD BANK,OH,451120,1997-02-28,1997,84,4,2.0,1,0,N,Y,60000.0,0.0,60000.0,48000.0
1,1000024006,NEW PARIS,IN,46526,1ST SOURCE BANK,IN,722410,1997-02-28,1997,60,2,2.0,1,0,N,Y,40000.0,0.0,40000.0,32000.0
2,1000034009,BLOOMINGTON,IN,47401,GRANT COUNTY STATE BANK,IN,621210,1997-02-28,1997,180,7,1.0,1,0,N,N,287000.0,0.0,287000.0,215250.0
3,1000044001,BROKEN ARROW,OK,74012,1ST NATL BK & TR CO OF BROKEN,OK,0,1997-02-28,1997,60,2,1.0,1,0,N,Y,35000.0,0.0,35000.0,28000.0
4,1000054004,ORLANDO,FL,32801,FLORIDA BUS. DEVEL CORP,FL,0,1997-02-28,1997,240,14,1.0,1,0,N,N,229000.0,0.0,229000.0,229000.0


In [13]:
## Lastly, we should be a bit suspicious of what we named the money_cols
## Why should there be three different numbers?  Perhaps there is the potential
## for additional data leakage that was not obvious from the feature description 
## From above it seems like "DisbursementGross" might just be a dupe of "GrAppv"
## So we can take a look at the interesting cases..
df.loc[(df["DisbursementGross"] != df["GrAppv"]), money_cols +["Defaulted"]].head()

,DisbursementGross,GrAppv,SBA_Appv,Defaulted
11,150000.0,300000.0,225000.0,0.0
28,438541.0,100000.0,50000.0,0.0
30,51440.0,35000.0,17500.0,0.0
34,63076.0,25000.0,12500.0,0.0
43,197485.0,200000.0,150000.0,1.0


In [14]:
## And now we see that indeed we have another leaky column!
## DisbursementGross indicated how much money was disbursed over the lifetime
## of the loan, rather than, something like how much was initially disbursed
## Therefore we should drop this column
df = df.drop(columns= "DisbursementGross", axis=1)
df.head()

,Id,City,State,Zip,Bank,BankState,NAICS,ApprovalDate,ApprovalFY,Term,NoEmp,NewExist,FranchiseCode,UrbanRural,RevLineCr,LowDoc,Defaulted,GrAppv,SBA_Appv
0,1000014003,EVANSVILLE,IN,47711,FIFTH THIRD BANK,OH,451120,1997-02-28,1997,84,4,2.0,1,0,N,Y,0.0,60000.0,48000.0
1,1000024006,NEW PARIS,IN,46526,1ST SOURCE BANK,IN,722410,1997-02-28,1997,60,2,2.0,1,0,N,Y,0.0,40000.0,32000.0
2,1000034009,BLOOMINGTON,IN,47401,GRANT COUNTY STATE BANK,IN,621210,1997-02-28,1997,180,7,1.0,1,0,N,N,0.0,287000.0,215250.0
3,1000044001,BROKEN ARROW,OK,74012,1ST NATL BK & TR CO OF BROKEN,OK,0,1997-02-28,1997,60,2,1.0,1,0,N,Y,0.0,35000.0,28000.0
4,1000054004,ORLANDO,FL,32801,FLORIDA BUS. DEVEL CORP,FL,0,1997-02-28,1997,240,14,1.0,1,0,N,N,0.0,229000.0,229000.0


In [15]:
## Lastly, we can turn "RevLineCr" and "LowDoc" into numeric types as well, since they are binary variables
## And for completeness (mainly for boolean compatability), we will modify NewExist,
## as currently it is either 1 or 2 rather than 0 or 1
## also, if we do not have info about the target variable it is not useful, so we drop

yes_no_cols = ["RevLineCr", "LowDoc"]
df[yes_no_cols] = df[yes_no_cols].apply(lambda col: col.map({'N':0,'Y':1}))

df["ExistingBusiness"] = df["NewExist"].apply(lambda x: x%2)
df = df.drop("NewExist", axis=1)

df = df.dropna(subset=["Defaulted"])
df.head()

,Id,City,State,Zip,Bank,BankState,NAICS,ApprovalDate,ApprovalFY,Term,NoEmp,FranchiseCode,UrbanRural,RevLineCr,LowDoc,Defaulted,GrAppv,SBA_Appv,ExistingBusiness
0,1000014003,EVANSVILLE,IN,47711,FIFTH THIRD BANK,OH,451120,1997-02-28,1997,84,4,1,0,0.0,1.0,0.0,60000.0,48000.0,0.0
1,1000024006,NEW PARIS,IN,46526,1ST SOURCE BANK,IN,722410,1997-02-28,1997,60,2,1,0,0.0,1.0,0.0,40000.0,32000.0,0.0
2,1000034009,BLOOMINGTON,IN,47401,GRANT COUNTY STATE BANK,IN,621210,1997-02-28,1997,180,7,1,0,0.0,0.0,0.0,287000.0,215250.0,1.0
3,1000044001,BROKEN ARROW,OK,74012,1ST NATL BK & TR CO OF BROKEN,OK,0,1997-02-28,1997,60,2,1,0,0.0,1.0,0.0,35000.0,28000.0,1.0
4,1000054004,ORLANDO,FL,32801,FLORIDA BUS. DEVEL CORP,FL,0,1997-02-28,1997,240,14,1,0,0.0,0.0,0.0,229000.0,229000.0,1.0


In [16]:
## And with that we are done with cleaning!
## Just need to save the cleaned data
## We split the data into a train set and a test set
from pathlib import Path
from sklearn.model_selection import train_test_split

target_col   = "Defaulted"
holdout_size = 0.10
df = df.copy()

dev_df, holdout_df = train_test_split(
    df,
    test_size=holdout_size,
    stratify=df[target_col],
    random_state=42,
    shuffle=True,
)

# Save them to disk
base = Path("/Users/stephenwillis/Desktop/archive")
base.mkdir(parents=True, exist_ok=True)

dev_path     = base / "cleaned_loan_data_dev.csv"
holdout_path = base / "cleaned_loan_data_holdout.csv"

dev_df.to_csv(dev_path, index=False)
holdout_df.to_csv(holdout_path, index=False)

print(f"\nSaved DEV → {dev_path}  |  HOLDOUT → {holdout_path}")
## Now time to move onto EDA


Saved DEV → /Users/stephenwillis/Desktop/archive/cleaned_loan_data_dev.csv  |  HOLDOUT → /Users/stephenwillis/Desktop/archive/cleaned_loan_data_holdout.csv
